# 02 · Limpieza y correcciones (S4)

Por cada slot de 12 min con los cuatro productos y `QUALITY == 0`:

1. **Continuo, oscurecimiento al limbo** (Castellanos Durán & Kleint 2020, Eqs. 1–2):
   $I_{corr} = I/C(\mu)$, con $C(\mu)=1-u-v+u\mu+v\mu^2$, $u=0.836$, $v=-0.204$ (Allen 1976, 6173 Å).
   Después se normaliza por el Sol tranquilo del propio frame, así que $I_c/I_{qs}\approx1$ fuera de la mancha.
2. **Dopplergrama** (Castellanos Durán, Lagg & Solanki 2021):
   $v = v_{LOS} - v_{SDO} - v_{LSF} - v_{CLV} - v_{grav}$
   - $v_{SDO}$: velocidad orbital de SDO, de las keywords OBS_VR, OBS_VW y OBS_VN.
   - $v_{LSF}$: rotación diferencial más flujo meridional.
   - $v_{CLV}$: blueshift convectivo, que depende de μ.
   - $v_{grav}$: redshift gravitacional, 636 m/s.
3. **Magnetograma**: B_LOS sin cambios (NaN fuera del disco), más el mapa de μ.
4. **SHARP**: $B_x=B_p$, $B_y=-B_t$, $B_z=B_r$, recortado a la zona del cutout. `conf_disambig` se guarda aparte. En el Sol tranquilo vale 50–60, porque la ambigüedad se resuelve con un método más simple; la umbra y la penumbra tienen 90.

Las funciones de corrección están copiadas tal cual de `sun_spots_pulsation`. Sus pruebas (`tests/`) pasan con los datos de NOAA 11536.

Salida: `data/interim/<id>/<producto>/<producto>_<YYYYMMDD_HHMMSS>.fits`, un FITS 2D con el WCS original, más `frames.csv` con los diagnósticos.

In [ ]:
import pathlib, sys, warnings
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

from src import config, cleaning

`OVERWRITE=False` salta los frames ya escritos. `MAX_FRAMES` limita los slots por mancha, para probar rápido.

In [ ]:
OVERWRITE = False
MAX_FRAMES = None
SPOTS = config.all_spots()   # o p. ej. [config.spot('L11536_20120731')]

In [ ]:
diags = {}
for s in SPOTS:
    try:
        diags[s['id']] = cleaning.clean_spot(s, overwrite=OVERWRITE, max_frames=MAX_FRAMES)
    except Exception as exc:
        print(f"{s['id']}: no se pudo limpiar — {exc!r}")

## Slots usables por mancha

Un slot se descarta si falta algún producto o si alguno tiene `QUALITY != 0`. Lo más común son eclipses de SDO, calibraciones o un frame que JSOC no entregó.

In [ ]:
summary = []
for s in SPOTS:
    f = config.INTERIM_DIR / s['id'] / 'frame_table.csv'
    if f.exists():
        t = pd.read_csv(f)
        summary.append(dict(id=s['id'], slots=len(t), usables=int(t['usable'].sum()),
                            procesados=len(diags.get(s['id'], []))))
pd.DataFrame(summary)

## Controles

- **Limb darkening**: ⟨C⟩ por frame tiene que seguir la posición de la mancha (más chico lejos del centro). La mediana del Sol tranquilo de $I_c/I_{qs}$ tiene que estar en ≈1.
- **Doppler**: cada término tiene que tener el tamaño esperado. $v_{SDO}$ es de unos km/s y oscila con el período orbital de 24 h; $v_{LSF}$ es de cientos de m/s y depende de la longitud; $v_{CLV}$ vale entre −280 y −370 m/s; $v_{grav}$ vale 636 m/s.
- La velocidad corregida del Sol tranquilo **no** queda en 0. Queda en unos −200 m/s, que es el sesgo absoluto conocido del Doppler de HMI. La prueba de cierre de `tests/test_doppler_calibration.py` mide −213 m/s en 11536. Lo que importa es que quede **plana** en el tiempo: la deriva instrumental de ±1 km/s debe desaparecer.

In [ ]:
ids = [i for i in diags if len(diags[i])]
fig, axes = plt.subplots(len(ids), 3, figsize=(16, 2.6 * len(ids)), squeeze=False, layout='constrained')
for row, sid in zip(axes, ids):
    d = diags[sid]
    row[0].plot(d.index, d['c_mean'], label='⟨C(μ)⟩')
    row[0].plot(d.index, d['mu_center'], '--', label='μ centro caja')
    row[0].set_ylabel(sid); row[0].legend(fontsize=8)
    row[1].plot(d.index, d['ic_quiet_median'], label='mediana Ic/Iqs Sol tranquilo')
    row[1].set_ylim(0.95, 1.05); row[1].legend(fontsize=8)
    for k in ('v_raw_quiet', 'v_corr_quiet', 'v_sdo', 'v_lsf', 'v_clv'):
        row[2].plot(d.index, d[k], label=k)
    row[2].legend(fontsize=7, ncol=2); row[2].set_ylabel('m/s')
    for ax in row:
        ax.tick_params(axis='x', labelrotation=30, labelsize=7)
plt.show()

## Un frame de ejemplo, antes y después

In [ ]:
import sunpy.map
from matplotlib.colors import Normalize
from src.download import PRODUCT_GLOBS
sid = ids[0]
s = config.spot(sid)
t = diags[sid].index[len(diags[sid]) // 2]
table = pd.read_csv(config.INTERIM_DIR / sid / 'frame_table.csv', index_col='time', parse_dates=['time'])
raw_ic = sunpy.map.Map(table.loc[t, 'continuum']); raw_v = sunpy.map.Map(table.loc[t, 'dopplergram'])
stamp = f'{t:%Y%m%d_%H%M%S}'
ic = sunpy.map.Map(config.INTERIM_DIR / sid / 'continuum' / f'continuum_{stamp}.fits')
v = sunpy.map.Map(config.INTERIM_DIR / sid / 'dopplergram' / f'dopplergram_{stamp}.fits')

fig = plt.figure(figsize=(16, 7), layout='constrained')
for i, (m, title, kw) in enumerate([
        (raw_ic, 'Ic crudo [DN/s]', dict(cmap='gray')),
        (ic, 'Ic/Iqs corregido', dict(cmap='gray', norm=Normalize(0.2, 1.1))),
        (raw_v, 'v_LOS cruda [m/s]', dict(cmap='RdBu_r')),
        (v, 'v corregida [m/s]', dict(cmap='RdBu_r', norm=Normalize(-1500, 1500)))]):
    ax = fig.add_subplot(2, 2, i + 1, projection=m)
    im = m.plot(axes=ax, title=title, **kw)
    plt.colorbar(im, ax=ax, fraction=0.03)
plt.show()